### RunnableParallel

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_teddynote import logging

logging.langsmith("LCEL-Advanced")

LangSmith 추적을 시작합니다.
[프로젝트명]
LCEL-Advanced


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming"], embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context: {context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

model=ChatOpenAI(model="gpt-4o-mini")

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("what is Teddy's occupation?")

"Teddy's occupation is an AI engineer."

In [9]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming"],
    embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain.invoke({"question": "What is Teddy's occupation?", "language": "Korean"})

"Teddy's occupation is an AI engineer."

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

model = ChatOpenAI()

capital_chain = (
    ChatPromptTemplate.from_template("{country} 의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain = (
    ChatPromptTemplate.from_template("{country} 의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

map_chain = RunnableParallel(capital=capital_chain, area=area_chain)

map_chain.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 서울입니다.', 'area': '대한민국의 면적은 100,363 제곱 킬로미터입니다.'}

In [11]:
capital_chain2 = (
    ChatPromptTemplate.from_template("{country1} 의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain2 = (
    ChatPromptTemplate.from_template("{country2} 의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

map_chain2 = RunnableParallel(capital=capital_chain2, area=area_chain2)

map_chain2.invoke({"country1": "영국", "country2": "캐나다"})

{'capital': '런던(london)입니다.', 'area': '캐나다의 총 면적은 약 9,984,670 제곱 킬로미터입니다.'}

In [12]:
%%timeit

area_chain.invoke({"country": "일본"})

1.28 s ± 489 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit

capital_chain.invoke({"country": "일본"})

1.1 s ± 200 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
%%timeit

map_chain.invoke({"country": "일본"})

1.83 s ± 665 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
